# 10 实践记录：扩充数据并开始 30 轮微调

本次把原 853 张 FMD 数据与 Humans in the Loop 的 Medical Mask 数据合并，进行更充分的训练。此笔记记录启动状态，最终结果以实验目录中的报告为准。

## 1. 新增数据来自哪里

- 完整标注版：[Humans in the Loop / Medical Mask](https://www.kaggle.com/datasets/humansintheloop/medical-mask-detection)。
- 发布者说明：[Medical mask dataset](https://humansintheloop.org/medical-mask-dataset/)。
- 同时下载并核对 Wobot 的早期版本，发现它们是同一批图片的不同发布版本，未重复当作独立新增数据计算。最终接受的新增样本均来自完整标注版。

图片的人工框和类别由原数据提供者标注。我们只做格式转换、类别映射、筛除与检查，没有把模型自动猜测当成人工标注。

## 2. 实际进入训练的数据

| 子集 | 图片数 | 正确戴口罩框 | 未戴口罩框 | 佩戴不正确框 |
|---|---:|---:|---:|---:|
| 扩充训练 | 4316 | 7614 | 1827 | 262 |
| 扩充验证 | 1081 | 1846 | 499 | 66 |

保留原 171 张验证图片，另加入 910 张新增验证图片。4 张与原验证图高度相似的旧训练图片被排除。

新增有效数据共 4548 张：3638 张训练、910 张验证。训练集中的“佩戴不正确”目标由原来的 98 个增加到 262 个。

含 face_other_covering 的图片整体排除，不把含义不明确的其他遮挡直接映射为现有三类；还排除无目标标注、无效框与重复/高度相似样本。全部 8060 个新增目标框已与源 JSON 独立核对。

转换后标签的检查图在项目 datasets/mask_expanded/previews/label-check.jpg。

## 3. 这次怎样训练

使用通用 yolov5s.pt 起点，输入 640、batch 16、workers 0、种子 42、SGD 默认超参数，运行 30 轮，不冻结网络参数。

开启 image-weights 类别加权抽样，使样本较少、当前表现较差的类别获得更多训练机会。抽样只作用于训练集，不改变验证集分布。

没有把所有图片缓存到内存。首轮训练部分约 4 分 25 秒，整个 30 轮可能需要两小时以上，实际时间会变化。

本轮同时增加数据、轮次，并改变 batch 和抽样，因此不能仅凭指标变化就断言提升全部来自更多数据。

## 4. 训练结束后会自动做什么

实验目录为项目 runs/train/mask_expanded_30epochs/。

- experiment-status.json：每 30 秒更新状态、已完成轮次及最近指标。
- weights/best.pt、weights/last.pt：训练中的检查点，运行结束前仍会更新。
- results.csv：逐轮训练和验证指标。
- 训练结束后自动评估扩充验证集、原 171 张验证集，以及同设置下的旧 3 轮权重。
- 最终生成 evaluation.json、training-summary.json、REPORT.md 与 verification/ 检测样例。

如果训练或评估失败，状态文件会记录错误。本次没有自动替换网页默认模型。

这份笔记中的“已启动”不代表 30 轮已全部完成。